<a href="https://colab.research.google.com/github/ayauuu/medqa-lora-finetune/blob/main/finetunething.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U transformers==4.46.3 peft==0.13.2 trl==0.12.1 accelerate==1.1.1 datasets==3.1.0
!pip install -q -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 26.9 MB/s eta 0:00:00


In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

GPU available: True
GPU name: Tesla T4


In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

print("Model loaded successfully")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully


In [ ]:
prompt = "What are the common symptoms of type 2 diabetes?"
messages = [{"role": "user", "content": prompt}]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True
).to(model.device)

output = model.generate(**inputs, max_new_tokens=200)
print(tokenizer.decode(output[0], skip_special_tokens=True))

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
What are the common symptoms of type 2 diabetes?
assistant
Type 2 diabetes is a chronic disease characterized by high blood sugar levels over time due to problems with insulin production or action in the body. The following are some common symptoms:

1. Frequent urination: This can be one of the earliest signs of diabetes.

2. Excessive thirst and increased water intake: When your body has difficulty using its cells for energy (due to low insulin), it will try to draw more fluid from the bloodstream into the urine to get rid of excess glucose.

3. Unexplained weight loss: Insufficient insulin can lead to the breakdown of muscle and fat tissue, which results in a decrease in body mass.

4. Fatigue: Your body may feel tired because there isn't enough fuel to meet your needs.

5. Blurred vision: High blood sugar can cause fluid to leak out of the tiny blood vessels in the retina of the eye, causing blurred vi

In [ ]:
from datasets import load_dataset

dataset = load_dataset("openlifescienceai/medmcqa", split="train")
print(dataset)
print(dataset[0])

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/85.9M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/936k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/1.48M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/182822 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6150 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4183 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'question', 'opa', 'opb', 'opc', 'opd', 'cop', 'choice_type', 'exp', 'subject_name', 'topic_name'],
    num_rows: 182822
})
{'id': 'e9ad821a-c438-4965-9f77-760819dfa155', 'question': 'Chronic urethral obstruction due to benign prismatic hyperplasia can lead to the following change in kidney parenchyma', 'opa': 'Hyperplasia', 'opb': 'Hyperophy', 'opc': 'Atrophy', 'opd': 'Dyplasia', 'cop': 2, 'choice_type': 'single', 'exp': 'Chronic urethral obstruction because of urinary calculi, prostatic hyperophy, tumors, normal pregnancy, tumors, uterine prolapse or functional disorders cause hydronephrosis which by definition is used to describe dilatation of renal pelvis and calculus associated with progressive atrophy of the kidney due to obstruction to the outflow of urine Refer Robbins 7yh/9,1012,9/e. P950', 'subject_name': 'Anatomy', 'topic_name': 'Urinary tract'}


In [ ]:
import random

def format_example(row):
    options = {"A": row["opa"], "B": row["opb"], "C": row["opc"], "D": row["opd"]}
    correct_letter = ["A", "B", "C", "D"][row["cop"]]
    correct_text = options[correct_letter]

    question_block = (
        f"{row['question']}\n"
        f"A. {row['opa']}\n"
        f"B. {row['opb']}\n"
        f"C. {row['opc']}\n"
        f"D. {row['opd']}"
    )

    explanation = row["exp"] if row["exp"] else "No explanation provided."
    answer_block = f"The correct answer is {correct_letter}. {correct_text}.\n\nExplanation: {explanation}"

    messages = [
        {"role": "user", "content": question_block},
        {"role": "assistant", "content": answer_block},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

# Subsample for a fast first training run — 3000 examples is plenty to see real results
small_dataset = dataset.shuffle(seed=42).select(range(3000))
formatted_dataset = small_dataset.map(format_example, remove_columns=small_dataset.column_names)

print(formatted_dataset[0]["text"])

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
OPV can be used if vaccine l monitor is showing?
A. Colour of outer circle is same as inner square
B. Colour of outer circle is darker than inner square
C. Colour of outer circle is lighter than inner square
D. None of the above<|im_end|>
<|im_start|>assistant
The correct answer is B. Colour of outer circle is darker than inner square.

Explanation: Ans. is 'b' i.e., Colour of outer circle is darker than inner square<|im_end|>



In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./medqa-lora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,
    report_to="none",
    max_seq_length=512,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    args=training_args,
)

trainer.train()

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,2.340600
20,1.894700
30,1.799600
40,1.797600
50,1.802800
60,1.843700
70,1.808400
80,1.639700
90,1.694600
100,1.762100


TrainOutput(global_step=375, training_loss=1.715532938639323, metrics={'train_runtime': 3884.0363, 'train_samples_per_second': 0.772, 'train_steps_per_second': 0.097, 'total_flos': 6115857642018816.0, 'train_loss': 1.715532938639323, 'epoch': 1.0})

In [ ]:
prompt = "What are the common symptoms of type 2 diabetes?"
messages = [{"role": "user", "content": prompt}]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True
).to(model.device)

output = model.generate(**inputs, max_new_tokens=200)
print(tokenizer.decode(output[0], skip_special_tokens=True))

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
What are the common symptoms of type 2 diabetes?
assistant
The following are some of the common symptoms of type 2 diabetes: 1) Polydipsia (excessive thirst and urination) 2) Polyuria (excessive urine production) 3) Weight loss or weight gain (independent of dieting habits) 4) Fatigue 5) Blurred vision 6) Irritability 7) Slow healing wounds 8) Tingling sensations in hands and feet


In [ ]:
test_row = dataset[5000]  # pick one it hasn't seen (outside our 3000-example training slice, roughly)
question_block = (
    f"{test_row['question']}\n"
    f"A. {test_row['opa']}\n"
    f"B. {test_row['opb']}\n"
    f"C. {test_row['opc']}\n"
    f"D. {test_row['opd']}"
)
messages = [{"role": "user", "content": question_block}]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)
output = model.generate(**inputs, max_new_tokens=200)
print(tokenizer.decode(output[0], skip_special_tokens=True))

correct_letter = ["A", "B", "C", "D"][test_row["cop"]]
print("\n--- Ground truth ---")
print(f"Correct answer: {correct_letter}. {test_row[['opa','opb','opc','opd'][test_row['cop']]]}")

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Binswanger&;s disease is a form of
A. Hypeensive retinopathy
B. Hypeensive nephropathy
C. Hypeensive encephalopathy
D. Subcoical leukoencephalopathy
assistant
The correct answer is C. Hypeensive encephalopathy.

Explanation: No explanation provided.

--- Ground truth ---
Correct answer: D. Subcoical leukoencephalopathy


In [ ]:
import re

eval_set = dataset.shuffle(seed=123).select(range(100))  # 100 fresh questions, not used in training
correct = 0
results = []

for row in eval_set:
    question_block = (
        f"{row['question']}\n"
        f"A. {row['opa']}\n"
        f"B. {row['opb']}\n"
        f"C. {row['opc']}\n"
        f"D. {row['opd']}"
    )
    messages = [{"role": "user", "content": question_block}]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)
    output = model.generate(**inputs, max_new_tokens=60, do_sample=False)
    response = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    match = re.search(r"correct answer is ([ABCD])", response, re.IGNORECASE)
    predicted = match.group(1).upper() if match else None
    correct_letter = ["A", "B", "C", "D"][row["cop"]]

    is_correct = predicted == correct_letter
    correct += is_correct
    results.append({"question": row["question"], "predicted": predicted, "correct": correct_letter, "is_correct": is_correct})

accuracy = correct / len(eval_set)
print(f"Accuracy: {accuracy:.2%} ({correct}/{len(eval_set)})")

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


Accuracy: 48.00% (48/100)


In [ ]:
with model.disable_adapter():
    correct_base = 0
    for row in eval_set:
        question_block = (
            f"{row['question']}\n"
            f"A. {row['opa']}\n"
            f"B. {row['opb']}\n"
            f"C. {row['opc']}\n"
            f"D. {row['opd']}"
        )
        messages = [{"role": "user", "content": question_block}]
        inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)
        output = model.generate(**inputs, max_new_tokens=60, do_sample=False)
        response = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

        match = re.search(r"correct answer is ([ABCD])", response, re.IGNORECASE)
        predicted = match.group(1).upper() if match else None
        correct_letter = ["A", "B", "C", "D"][row["cop"]]
        correct_base += predicted == correct_letter

    accuracy_base = correct_base / len(eval_set)
    print(f"Base model accuracy: {accuracy_base:.2%} ({correct_base}/{len(eval_set)})")

Base model accuracy: 9.00% (9/100)


In [ ]:
none_count = 0
with model.disable_adapter():
    for row in eval_set:
        question_block = (
            f"{row['question']}\n"
            f"A. {row['opa']}\n"
            f"B. {row['opb']}\n"
            f"C. {row['opc']}\n"
            f"D. {row['opd']}"
        )
        messages = [{"role": "user", "content": question_block}]
        inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)
        output = model.generate(**inputs, max_new_tokens=60, do_sample=False)
        response = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        match = re.search(r"correct answer is ([ABCD])", response, re.IGNORECASE)
        if match is None:
            none_count += 1

print(f"Unparseable responses: {none_count}/100")
# print one example so we can see what the base model actually says
print("\nExample raw response:")
print(response)

Unparseable responses: 86/100

Example raw response:
Given Ramachandran's condition, which involves a non-seminomatous tumor of the testis with involvement of more than four retroperitoneal lymph nodes, the appropriate treatment options would be:

- **Inguinal orchiectomy**: This is typically performed to remove the testicle if


In [ ]:
!git clone https://github.com/ayauuu/medqa-lora-finetune.git
%cd medqa-lora-finetune

Cloning into 'medqa-lora-finetune'...
/content/medqa-lora-finetune


In [ ]:
%%writefile README.md
# Medical QA — LoRA Fine-Tune

Fine-tuning Qwen2.5-1.5B-Instruct on MedMCQA (medical exam Q&A) using QLoRA on a free Colab T4 GPU.

## Status
🚧 In progress

## Method
- Base model: Qwen2.5-1.5B-Instruct
- Dataset: MedMCQA (openlifescienceai/medmcqa), 3000-example training subset
- Fine-tuning: QLoRA (4-bit, LoRA rank=16) via PEFT + TRL
- Trained 1 epoch, final training loss ~1.72

## Results (100 held-out questions)
- Fine-tuned model accuracy: **48%**
- Base model accuracy: 9% raw, but 86/100 of its answers didn't even follow the required answer format — so this isn't a fair comparison of medical knowledge, it mainly shows fine-tuning taught the model to answer in the expected structured format reliably, in addition to improving correctness.

## Next steps
- [ ] Retrain and immediately save adapter to Hugging Face Hub (avoid losing it to a runtime disconnect)
- [ ] Deploy a demo via Hugging Face Spaces (Gradio)
- [ ] Expand evaluation set

Writing README.md


In [ ]:
!git add .
!git commit -m "Add README with initial results"
!git push

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@f3e8929532cb.(none)')
error: src refspec refs/heads/main does not match any
error: failed to push some refs to 'https://github.com/ayauuu/medqa-lora-finetune.git'


In [ ]:
!git add .

In [ ]:
!git commit -m "Add README with initial results"


Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@f3e8929532cb.(none)')


In [ ]:
!git push

error: src refspec refs/heads/main does not match any
error: failed to push some refs to 'https://github.com/ayauuu/medqa-lora-finetune.git'


In [ ]:
!git config --global user.email "abbassi.ayabu@gmail.com"
!git config --global user.name "ayauuu"

In [ ]:
!git add .
!git commit -m "Add README with initial results"
!git push

[main (root-commit) ea2dcf9] Add README with initial results
 1 file changed, 21 insertions(+)
 create mode 100644 README.md
fatal: could not read Username for 'https://github.com': No such device or address


In [ ]:
!git add .

In [ ]:
!git commit -m "Add README with initial results"


On branch main
Your branch is based on 'origin/main', but the upstream is gone.
  (use "git branch --unset-upstream" to fixup)

nothing to commit, working tree clean


In [ ]:
!git push

fatal: could not read Username for 'https://github.com': No such device or address


In [ ]:
!git remote set-url origin https://4neQ3ByBK1tN2otHzuBagC9M8Tx2MXgmWgCv3ufV1Zz6PQG63OCYzd56NBn@github.com/ayauuu/medqa-lora-finetune.git

In [ ]:
!git push

fatal: could not read Password for 'https://4neQ3ByBK1tN2otHzuBagC9M8Tx2MXgmWgCv3ufV1Zz6PQG63OCYzd56NBn@github.com': No such device or address


In [ ]:
!git remote -v

origin	https://4neQ3ByBK1tN2otHzuBagC9M8Tx2MXgmWgCv3ufV1Zz6PQG63OCYzd56NBn@github.com/ayauuu/medqa-lora-finetune.git (fetch)
origin	https://4neQ3ByBK1tN2otHzuBagC9M8Tx2MXgmWgCv3ufV1Zz6PQG63OCYzd56NBn@github.com/ayauuu/medqa-lora-finetune.git (push)


In [ ]:
token = "4neQ3ByBK1tN2otHzuBagC9M8Tx2MXgmWgCv3ufV1Zz6PQG63OCYzd56NBn"  # e.g. ghp_xxxxxxxxxxxx
username = "ayauuu"
repo = "medqa-lora-finetune"

import subprocess
url = f"https://{token}@github.com/{username}/{repo}.git"
subprocess.run(["git", "remote", "set-url", "origin", url], check=True)
print("Remote updated")

Remote updated


In [ ]:
!git push

fatal: could not read Password for 'https://4neQ3ByBK1tN2otHzuBagC9M8Tx2MXgmWgCv3ufV1Zz6PQG63OCYzd56NBn@github.com': No such device or address


In [ ]:
print(len(token))
print(token[:4])

59
4neQ


In [21]:
from getpass import getpass
import requests

token = getpass("Paste your GitHub token: ")
r = requests.get("https://api.github.com/user", headers={"Authorization": f"token {token}"})
print(r.status_code)
print(r.json())

Paste your GitHub token: ··········
200
{'login': 'ayauuu', 'id': 170805745, 'node_id': 'U_kgDOCi5J8Q', 'avatar_url': 'https://avatars.githubusercontent.com/u/170805745?v=4', 'gravatar_id': '', 'url': 'https://api.github.com/users/ayauuu', 'html_url': 'https://github.com/ayauuu', 'followers_url': 'https://api.github.com/users/ayauuu/followers', 'following_url': 'https://api.github.com/users/ayauuu/following{/other_user}', 'gists_url': 'https://api.github.com/users/ayauuu/gists{/gist_id}', 'starred_url': 'https://api.github.com/users/ayauuu/starred{/owner}{/repo}', 'subscriptions_url': 'https://api.github.com/users/ayauuu/subscriptions', 'organizations_url': 'https://api.github.com/users/ayauuu/orgs', 'repos_url': 'https://api.github.com/users/ayauuu/repos', 'events_url': 'https://api.github.com/users/ayauuu/events{/privacy}', 'received_events_url': 'https://api.github.com/users/ayauuu/received_events', 'type': 'User', 'user_view_type': 'public', 'site_admin': False, 'name': 'Aya', 'com

In [22]:
import subprocess

username = "ayauuu"
repo = "medqa-lora-finetune"

url = f"https://{token}@github.com/{username}/{repo}.git"
subprocess.run(["git", "remote", "set-url", "origin", url], check=True)

result = subprocess.run(["git", "push"], capture_output=True, text=True)
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr)

STDOUT: 
STDERR: To https://github.com/ayauuu/medqa-lora-finetune.git
 * [new branch]      main -> main

